In [96]:
# ============================================================
# Рисунок 3.2 — Преобразование накопительных продаж
# в квартальные значения
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/gdrive")
except Exception:
    pass

# ------------------------------------------------------------
# 1. Загрузка данных
# ------------------------------------------------------------

DATA_PATH = Path("/content/gdrive/MyDrive/rostat/group_quarterly_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Файл не найден: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["period_end"])
df["period_end"] = pd.to_datetime(df["period_end"])

# ------------------------------------------------------------
# 2. Параметры графика
# ------------------------------------------------------------

GROUP_CODE = "G01"
YEAR_TO_PLOT = 2025

g_year = (
    df[
        (df["product_group_code"] == GROUP_CODE) &
        (df["year"] == YEAR_TO_PLOT)
    ]
    .sort_values("quarter")
    .copy()
)

if g_year.empty:
    raise ValueError(f"Нет данных для группы {GROUP_CODE} за {YEAR_TO_PLOT} год.")

required_cols = [
    "period_end",
    "quarter",
    "product_group_name",
    "sales_quarter_thousand_rub",
    "sales_cumulative_thousand_rub",
]

missing = [c for c in required_cols if c not in g_year.columns]

if missing:
    raise ValueError(f"В датасете нет нужных колонок: {missing}")

group_name = g_year["product_group_name"].iloc[0]
g_year["quarter_label"] = "Q" + g_year["quarter"].astype(str)

# ------------------------------------------------------------
# 3. Построение графика
# ------------------------------------------------------------

fig, ax1 = plt.subplots(figsize=(12, 6))

# Столбцы: продажи за отдельный квартал
bars = ax1.bar(
    g_year["quarter_label"],
    g_year["sales_quarter_thousand_rub"],
    alpha=0.65,
    label="Продажи за отдельный квартал"
)

ax1.set_xlabel("Квартал")
ax1.set_ylabel("Продажи за квартал, тыс. руб.")
ax1.grid(axis="y", alpha=0.3)

# Подписи значений над столбцами
for bar, value in zip(bars, g_year["sales_quarter_thousand_rub"]):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,.0f}".replace(",", " "),
        ha="center",
        va="bottom",
        fontsize=9
    )

# Линия: накопительные продажи с начала года
ax2 = ax1.twinx()

ax2.plot(
    g_year["quarter_label"],
    g_year["sales_cumulative_thousand_rub"],
    marker="o",
    linewidth=2.5,
    label="Накопительные продажи с начала года"
)

ax2.set_ylabel("Накопительные продажи, тыс. руб.")

# Заголовок
plt.title(
    f"Преобразование накопительных продаж в квартальные значения\n"
    f"{group_name}, {YEAR_TO_PLOT} год"
)

# Общая легенда для двух осей
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines_1 + lines_2,
    labels_1 + labels_2,
    loc="upper left"
)

plt.tight_layout()

# ------------------------------------------------------------
# 4. Сохранение рисунка
# ------------------------------------------------------------

OUT_DIR = DATA_PATH.parent / "plots_for_diploma"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / f"рисунок_3_2_преобразование_накопительных_продаж_{GROUP_CODE}_{YEAR_TO_PLOT}.png"

plt.savefig(OUT_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Рисунок сохранён:", OUT_PATH)

FileNotFoundError: Файл не найден: /content/gdrive/MyDrive/rostat/group_quarterly_dataset.csv

FileNotFoundError: Не найден файл group_quarterly_dataset.csv внутри /content/gdrive/MyDrive/rostat